# Going Deeper
## Advanced Extensions for Fast Finishers

**Vanderbilt Programs for Talented Youth | Day 4 | June 12, 2026**

---

This notebook has one hard problem for each group. Each problem requires reasoning, not just
running code. Some have no single correct answer. If you finish your group's main extension
notebook and still have time, find your group's section below and work through it.

You do not need to complete all sections. Read the one for your group.

---

**Sections:**

1. sPHENIX / RHIC -- Path-length dependence and surface bias
2. ALICE / LHC -- Why does the fireball freeze out at the phase boundary?
3. ATLAS / LHC -- How many events does it take to make a discovery?
4. IceCube -- Neutrino oscillations and what arrives at the detector
5. LBL Cyclotron -- Why does the island of stability exist?

---

---

## Section 1 (sPHENIX Group): Path-Length Dependence and Surface Bias

In your xJ notebook, you saw that the subleading jet is more suppressed than the leading jet
on average. This seems puzzling at first: both jets are produced in the same collision, so why
does one lose more energy than the other?

The answer has to do with geometry.

### The setup

When two gold nuclei collide, the QGP fireball is roughly circular in the transverse plane for
central collisions. Hard scatterings that produce back-to-back dijets can happen anywhere inside
the overlap region.

The key insight is this: a jet produced near the surface of the fireball has a short path through
the hot matter and loses little energy. A jet produced near the center has a long path and loses
a lot. Because the two jets in a pair travel in opposite directions, if one jet is heading toward
the surface (short path), the other is heading away from it (long path).

This is called **surface bias**: the leading jet (which we require to have high pT) is biased
toward having taken a short path through the medium, which means the subleading jet is biased
toward having taken a long path.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(99)

R_fireball = 1.0
N_events   = 20000

r_prod   = R_fireball * np.sqrt(np.random.uniform(0, 1, N_events))
phi_prod = np.random.uniform(0, 2*np.pi, N_events)
x_prod   = r_prod * np.cos(phi_prod)
y_prod   = r_prod * np.sin(phi_prod)
phi_jet  = np.random.uniform(0, 2*np.pi, N_events)

def path_length(x0, y0, phi, R):
    b = 2 * (x0*np.cos(phi) + y0*np.sin(phi))
    c = x0**2 + y0**2 - R**2
    disc = b**2 - 4*c
    t = (-b + np.sqrt(np.maximum(disc, 0))) / 2
    return np.maximum(t, 0)

L1 = path_length(x_prod, y_prod, phi_jet,         R_fireball)
L2 = path_length(x_prod, y_prod, phi_jet + np.pi, R_fireball)

epsilon    = 0.4
pT_initial = 30.0

pT1 = pT_initial * np.exp(-epsilon * L1)
pT2 = pT_initial * np.exp(-epsilon * L2)

mask_selected = pT1 >= pT2
xJ_selected   = pT2[mask_selected] / pT1[mask_selected]
L1_selected   = L1[mask_selected]
L2_selected   = L2[mask_selected]

print(f'Events passing leading-jet selection: {mask_selected.sum():,} / {N_events:,}')
print(f'Mean path length, leading jet:    {L1_selected.mean():.3f} R')
print(f'Mean path length, subleading jet: {L2_selected.mean():.3f} R')
print(f'Mean xJ after selection: {xJ_selected.mean():.3f}')
print()
print('If there were no surface bias, both path lengths would be equal.')
print('The selection forces L1 < L2 on average -- this is surface bias.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.hist(L1_selected, bins=40, color='steelblue', alpha=0.7,
        density=True, label='Leading jet path length')
ax.hist(L2_selected, bins=40, color='firebrick', alpha=0.7,
        density=True, label='Subleading jet path length')
ax.set_xlabel('Path length through QGP (R units)', fontsize=12)
ax.set_ylabel('Normalized yield', fontsize=12)
ax.set_title('Surface bias: path lengths after leading-jet selection', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax = axes[1]
sc = ax.scatter(x_prod[mask_selected], y_prod[mask_selected],
                c=L1_selected, cmap='Blues', s=1, alpha=0.3)
theta_c = np.linspace(0, 2*np.pi, 200)
ax.plot(R_fireball*np.cos(theta_c), R_fireball*np.sin(theta_c), 'white', linewidth=1.5)
ax.set_aspect('equal')
ax.set_facecolor('#0D1117')
ax.set_title('Production points: shorter leading path = brighter', fontsize=11)
ax.set_xlabel('x (fireball units)', fontsize=12)
ax.set_ylabel('y (fireball units)', fontsize=12)

ax = axes[2]
bins = np.linspace(0.3, 1.0, 29)
for eps, col, lbl in [(0.1,'steelblue','Weak quenching'),
                       (0.4,'darkorange','Medium quenching'),
                       (0.8,'firebrick','Strong quenching')]:
    pt1 = pT_initial * np.exp(-eps * L1)
    pt2 = pT_initial * np.exp(-eps * L2)
    mk  = pt1 >= pt2
    xj  = (pt2/pt1)[mk]
    cnt, edges = np.histogram(xj, bins=bins, density=True)
    cen = 0.5*(edges[:-1]+edges[1:])
    ax.plot(cen, cnt, color=col, linewidth=2, label=lbl)
ax.set_xlabel('xJ = pT2 / pT1', fontsize=12)
ax.set_ylabel('Normalized yield', fontsize=12)
ax.set_title('xJ distribution vs quenching strength', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Path-length dependence of jet energy loss: surface bias geometry',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Questions

1. After applying the leading-jet selection (requiring pT1 >= pT2), compare the mean path
   lengths of the leading and subleading jets. Which is longer? Explain why this must be true
   given how the selection works.

2. Look at the production point scatter plot. Are selected events concentrated near the surface
   of the fireball or near the center? Explain why in terms of the selection requirement.

3. As you increase the quenching strength (epsilon), what happens to the xJ distribution? Does
   it shift to higher or lower values? Why does stronger quenching push xJ lower?

4. In a perfectly central Au+Au collision (zero impact parameter, perfectly circular overlap),
   would you still expect surface bias? What about in a very peripheral collision where the
   overlap region is very small?

5. Real sPHENIX measurements can test path-length dependence by measuring xJ as a function of
   the azimuthal angle between the jet axis and the reaction plane (the plane containing the
   two beam directions and the impact parameter vector). Why would rotating the dijet pair
   relative to the reaction plane change the effective path lengths?

---

## Section 2 (ALICE Group): Why Does the Fireball Freeze Out at the Phase Boundary?

In your hadron ratios notebook, you found that the chemical freeze-out temperature T_chem is
approximately 156 MeV, which is strikingly close to the QCD phase transition temperature T_c
of about 154-158 MeV from lattice QCD calculations.

This agreement is not obvious. It could have been that T_chem was much lower than T_c (the
system cools significantly before freezing out) or much higher (freeze-out happens during the
QGP phase itself). Instead, T_chem sits right at the phase boundary. Why?

### The physical argument

Think about what chemical freeze-out means. It is the temperature at which inelastic collisions
(processes that change particle species) become rare enough that the species ratios stop changing.
In the QGP phase, quarks and gluons can freely change flavor through interactions like
q + qbar -> g + g -> s + sbar. These processes happen at rates proportional to the strong
coupling alpha_s.

As the system cools and crosses T_c, quarks and gluons become confined into hadrons. The
inelastic processes that can still change species ratios are now hadron-hadron reactions like
pi + pi -> K + Kbar. These reactions have much smaller cross-sections than QGP processes,
because the relevant coupling at hadronic energies is suppressed.

The argument is that once the system crosses T_c, the inelastic reaction rate drops so
precipitously that chemical equilibrium is immediately frozen in.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T   = np.linspace(100, 250, 500)
T_c = 156.0
m_pi = 140.0

alpha_s_qgp = 0.3 * (T/T_c)**(-0.5)
alpha_s_had = 0.3 * np.ones_like(T)

rate_qgp   = alpha_s_qgp**2 * (T/T_c)**3
rate_had   = alpha_s_had**2 * (T/T_c)**3 * np.exp(-m_pi/T) * 0.05
rate_total = np.where(T >= T_c, rate_qgp, rate_had)
rate_exp   = 0.15 * (T/T_c)**2

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(T, rate_qgp,   color='firebrick', linewidth=2.5, linestyle='--',
            label='QGP inelastic rate (if no phase transition)', alpha=0.5)
ax.semilogy(T, rate_had,   color='steelblue', linewidth=2.5, linestyle='--',
            label='Hadronic inelastic rate', alpha=0.5)
ax.semilogy(T, rate_total, color='white',     linewidth=3,
            label='Actual rate (QGP above T_c, hadronic below)')
ax.semilogy(T, rate_exp,   color='gold',      linewidth=2, linestyle=':',
            label='Fireball expansion rate')
ax.axvline(x=T_c, color='cyan', linewidth=1.5, linestyle=':', alpha=0.7)
ax.text(T_c+2, 0.001, 'T_c = 156 MeV', color='cyan', fontsize=11)

cross_idx = np.argmin(np.abs(rate_total[T < T_c] - rate_exp[T < T_c]))
T_fo = T[T < T_c][cross_idx]
ax.axvline(x=T_fo, color='gold', linewidth=1.5, linestyle='--', alpha=0.7)
ax.text(T_fo-18, 0.0002, f'T_freeze-out ~ {T_fo:.0f} MeV', color='gold', fontsize=10)

ax.set_xlabel('Temperature (MeV)', fontsize=13)
ax.set_ylabel('Reaction rate (schematic, arb. units)', fontsize=13)
ax.set_title('Why chemical freeze-out occurs near T_c\n'
             'The phase transition causes a sudden drop in inelastic reaction rates', fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.2, which='both')
ax.set_xlim(100, 250)
plt.tight_layout()
plt.show()

print(f'Schematic freeze-out temperature: {T_fo:.0f} MeV')
print(f'Real measured T_chem: ~156 MeV')
print(f'QCD T_c (lattice): ~154-158 MeV')

### Questions

1. Look at the rate plot. At T_c, the total rate drops by a large factor. What physical process
   causes this sudden drop? (Hint: think about what changes when quarks and gluons become
   confined into hadrons.)

2. The hadronic rate has a Boltzmann suppression factor exp(-m_pi / T). At T = 140 MeV, what
   is this factor numerically? What does it mean physically?

3. If the pion mass were much smaller (say 20 MeV instead of 140 MeV), would you expect T_chem
   to be closer to T_c or further from it? Explain using the rate plot logic.

4. ALICE measures particle ratios in Pb+Pb collisions at much higher energy than RHIC. The
   extracted T_chem is approximately the same at LHC and RHIC (both near 156 MeV). What does
   this universality tell you about the QCD phase boundary?

5. The model here treats the expansion rate as a smooth power law. In reality the fireball
   expands very rapidly at early times (nearly explosive). How would a faster expansion rate
   change the freeze-out temperature, and why?

---

## Section 3 (ATLAS Group): How Many Events Does It Take to Make a Discovery?

You found a Higgs signal in your notebook with some statistical significance. But the real ATLAS
discovery required 5-sigma -- the probability of the result being a statistical fluctuation is
less than 1 in 3.5 million. Let us work out exactly what determines significance and how to
reach 5-sigma.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(2012)

m_bins    = np.linspace(100, 160, 61)
m_centers = 0.5 * (m_bins[:-1] + m_bins[1:])
m_higgs   = 125.09

def background(m, N_bkg, a, b):
    return N_bkg * np.exp(a*(m-100) + b*(m-100)**2)

def gaussian(m, N, mu, sigma):
    return N * np.exp(-0.5*((m-mu)/sigma)**2)

def run_experiment(N_signal, N_bkg, sigma_det=1.8, seed=None):
    if seed is not None:
        np.random.seed(seed)
    bkg_rate = background(m_centers, N_bkg, -0.04, -0.0002)
    bkg_rate = bkg_rate / bkg_rate.sum() * N_bkg
    sig_rate = N_signal * np.exp(-0.5*((m_centers-m_higgs)/sigma_det)**2)
    sig_rate /= sig_rate.sum()
    sig_rate *= N_signal
    observed     = np.random.poisson(bkg_rate + sig_rate)
    observed_err = np.sqrt(np.maximum(observed, 1))
    sb_mask = (m_centers < 120) | (m_centers > 130)
    try:
        popt, _ = curve_fit(background, m_centers[sb_mask], observed[sb_mask],
                            p0=[N_bkg, -0.04, -0.0002],
                            sigma=observed_err[sb_mask], absolute_sigma=True)
        residuals = observed - background(m_centers, *popt)
        sig_mask  = (m_centers >= 115) & (m_centers <= 135)
        popt_s, pcov_s = curve_fit(gaussian, m_centers[sig_mask], residuals[sig_mask],
                                    p0=[N_signal*0.8, 125, 2.0],
                                    sigma=observed_err[sig_mask], absolute_sigma=True)
        return popt_s[0] / np.sqrt(pcov_s[0,0])
    except Exception:
        return 0

print('Significance vs number of Higgs signal events (background fixed at 8000):')
print(f'{"N_signal":>10}  {"Median sig.":>12}  {"Reach 5-sigma?":>15}')
print('-' * 45)

signal_counts = [50, 100, 200, 350, 500, 750, 1000]
median_sigs   = []
for N_sig in signal_counts:
    sigs = [run_experiment(N_sig, 8000, seed=s) for s in range(5)]
    med  = float(np.median(sigs))
    median_sigs.append(med)
    flag = 'YES' if med >= 5.0 else 'no'
    print(f'{N_sig:>10}  {med:>12.1f}  {flag:>15}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(signal_counts, median_sigs, 'o-', color='steelblue', linewidth=2.5, markersize=8)
ax.axhline(y=5.0, color='gold',       linewidth=2,   linestyle='--', label='5-sigma discovery threshold')
ax.axhline(y=3.0, color='darkorange', linewidth=1.5, linestyle=':',  label='3-sigma evidence threshold')
ax.set_xlabel('Number of Higgs signal events', fontsize=12)
ax.set_ylabel('Median significance (sigma)', fontsize=12)
ax.set_title('How much signal do we need for a discovery?', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

print('Significance vs detector energy resolution (200 signal events, 8000 background):')
print(f'{"Resolution (GeV)":>18}  {"Median sig.":>12}  {"Reach 5-sigma?":>15}')
print('-' * 50)

resolutions   = [0.8, 1.0, 1.5, 1.8, 2.5, 3.5, 5.0]
med_sigs_res  = []
for sigma_d in resolutions:
    sigs = [run_experiment(200, 8000, sigma_det=sigma_d, seed=s) for s in range(5)]
    med  = float(np.median(sigs))
    med_sigs_res.append(med)
    flag = 'YES' if med >= 5.0 else 'no'
    print(f'{sigma_d:>18.1f}  {med:>12.1f}  {flag:>15}')

ax = axes[1]
ax.plot(resolutions, med_sigs_res, 's-', color='firebrick', linewidth=2.5, markersize=8)
ax.axhline(y=5.0, color='gold',       linewidth=2,   linestyle='--', label='5-sigma threshold')
ax.axhline(y=3.0, color='darkorange', linewidth=1.5, linestyle=':',  label='3-sigma threshold')
ax.axvline(x=1.8, color='steelblue',  linewidth=1.5, linestyle=':',  alpha=0.7, label='ATLAS lead/LAr (1.8 GeV)')
ax.set_xlabel('Detector energy resolution sigma (GeV)', fontsize=12)
ax.set_ylabel('Median significance (sigma)', fontsize=12)
ax.set_title('How much does detector resolution matter?', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Two paths to discovery: more data vs better detector', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Questions

1. From the first plot: roughly how many signal events are needed to reach 5-sigma with 8000
   background events? How does this compare to the 200 events in your main notebook?

2. Significance scales approximately as N_signal / sqrt(N_background) in the peak region.
   Using this formula, estimate how many times more data ATLAS would need to collect (compared
   to your 200-event simulation) to reach 5-sigma. Does your plot agree?

3. From the second plot: how much does improving the resolution from 1.8 GeV to 1.0 GeV improve
   the significance for the same 200 signal events? Why does better resolution help?

4. The two plots show two strategies: collect more data vs build a better detector. For a fixed
   total cost, which strategy is more effective? (There is no single right answer, but think
   about what each plot shows.)

5. Real ATLAS had approximately 5.9-sigma significance in 2012 with about 400 H->yy candidate
   events above background. Using your significance formula from Q2, back-calculate the
   effective background level in the peak region. Does this match your expectations?

---

## Section 4 (IceCube Group): Neutrino Oscillations

Your accelerator beam produces muon neutrinos (nu_mu) from pion decay. But by the time they
reach IceCube, the neutrinos you detect may not all be nu_mu. Neutrinos oscillate between
flavors as they travel.

Neutrinos have mass (tiny, but nonzero), and the three mass eigenstates (nu_1, nu_2, nu_3) are
not the same as the flavor eigenstates (nu_e, nu_mu, nu_tau). A neutrino produced as nu_mu is
a quantum superposition of mass eigenstates, and as it travels, the phase of each component
evolves differently. The interference between components causes the probability of detecting
each flavor to oscillate with distance.

The two-flavor survival probability is:

P(nu_mu -> nu_mu) = 1 - sin^2(2*theta) * sin^2(1.267 * delta_m^2 * L / E)

where theta is the mixing angle, delta_m^2 is the mass squared difference in eV^2, L is the
baseline in km, and E is the neutrino energy in GeV.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta_23 = np.pi / 4
dm2_32   = 2.5e-3

def survival_prob(L_km, E_GeV, theta, dm2):
    phase = 1.267 * dm2 * L_km / E_GeV
    return 1 - (np.sin(2*theta)**2) * (np.sin(phase)**2)

L_range  = np.linspace(0.1, 1000, 5000)
energies = [1.0, 3.0, 10.0, 30.0]
colors   = ['steelblue', 'darkorange', 'firebrick', 'mediumpurple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for E, col in zip(energies, colors):
    ax.plot(L_range, survival_prob(L_range, E, theta_23, dm2_32),
            color=col, linewidth=2, label=f'E = {E:.0f} GeV')
ax.set_xlabel('Baseline L (km)', fontsize=12)
ax.set_ylabel('P(nu_mu -> nu_mu)', fontsize=12)
ax.set_title('Muon neutrino survival probability vs distance\n'
             '(atmospheric mixing parameters)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

LoE    = np.linspace(0, 5000, 10000)
p_LoE  = 1 - (np.sin(2*theta_23)**2) * (np.sin(1.267 * dm2_32 * LoE)**2)

ax = axes[1]
ax.plot(LoE, p_LoE, color='steelblue', linewidth=1.5)
ax.set_xlabel('L/E (km/GeV)', fontsize=12)
ax.set_ylabel('P(nu_mu -> nu_mu)', fontsize=12)
ax.set_title('Oscillation as a function of L/E\n'
             '(the natural variable for oscillation physics)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
ax.set_xlim(0, 5000)

plt.suptitle('Neutrino oscillations: nu_mu survival probability', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

L_min = np.pi / (2 * 1.267 * dm2_32)
print(f'First oscillation minimum at L/E = {L_min:.0f} km/GeV')
print(f'For E = 1 GeV:  first minimum at L = {L_min:.0f} km')
print(f'For E = 10 GeV: first minimum at L = {L_min*10:.0f} km')

In [ ]:
print('Fraction of nu_mu surviving to IceCube for different baselines and energies:')
print()
print(f'{"Baseline (km)":>15}  {"Energy (GeV)":>14}  {"Survival prob.":>16}  Scenario')
print('-' * 72)

scenarios = [
    (1,     1.0,  'Surface accelerator, 1 km baseline'),
    (10,    1.0,  'Surface accelerator, 10 km baseline'),
    (500,   3.0,  'Long-baseline, MINOS-like'),
    (500,  10.0,  'Long-baseline, high energy'),
    (12700, 3.0,  'Earth diameter baseline'),
    (12700,10.0,  'Earth diameter, high energy'),
]
for L, E, label in scenarios:
    p = survival_prob(L, E, theta_23, dm2_32)
    print(f'{L:>15,}  {E:>14.1f}  {p:>16.3f}  {label}')

print()
print('For short baselines, most nu_mu survive.')
print('For long baselines (Earth-crossing), oscillation is significant.')

### Questions

1. From the survival probability plot: at what baseline distance does the nu_mu survival
   probability first reach its minimum for E = 1 GeV? For E = 10 GeV? What is the pattern?

2. The oscillation probability depends on L/E, not L and E separately. What does this mean
   for designing an experiment -- if you want to observe the oscillation minimum, what
   relationship between your baseline and your beam energy do you need?

3. If your accelerator beam has an energy of 3 GeV and you place your accelerator 500 km from
   IceCube, what fraction of nu_mu survive? What flavor are the rest? (In the two-flavor
   approximation used here, the oscillated flavor is nu_tau.)

4. IceCube is primarily sensitive to nu_mu (which produce long muon tracks) rather than nu_tau
   (which produce shower-like events). If a significant fraction of your beam has oscillated
   to nu_tau by the time it reaches IceCube, how does this affect your design? What baseline
   would you choose to maximize nu_mu at the detector?

5. The parameters used here (theta_23, delta_m^2_32) are the atmospheric oscillation
   parameters. There is also a solar sector (theta_12, delta_m^2_21) with different values.
   Would solar oscillations matter for a short-baseline accelerator experiment? Why or why not?

---

## Section 5 (LBL Group): Why Does the Island of Stability Exist?

The island of stability is a prediction from the nuclear shell model. In the simple liquid drop
model, nuclear binding energy depends smoothly on A and Z. But the real binding energy has
additional structure: certain magic numbers (2, 8, 20, 28, 50, 82, 126) give extra stability,
just as closed electron shells give extra stability to noble gases. The next predicted magic
numbers for protons (Z = 114 or 120) and neutrons (N = 184) would create a doubly magic
superheavy nucleus with enhanced stability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a_V =  15.75
a_S =  17.80
a_C =   0.711
a_A =  23.70

def delta_pair(A, Z):
    if A % 2 == 1:  return 0
    if Z % 2 == 0:  return  12/np.sqrt(A)
    return -12/np.sqrt(A)

def BE_ld(A, Z):
    N = A - Z
    return (a_V*A - a_S*A**(2/3) - a_C*Z*(Z-1)/A**(1/3)
            - a_A*(A-2*Z)**2/A + delta_pair(A, Z))

def shell_corr(Z, N):
    magic = [2, 8, 20, 28, 50, 82, 126]
    pred_p = [114, 120]
    pred_n = [184]
    c = 0.0
    for m in magic:
        if Z == m: c += 3.0
        if N == m: c += 3.0
    for m in pred_p:
        c += 2.0 * np.exp(-(Z-m)**2/(2*5**2))
    for m in pred_n:
        c += 2.0 * np.exp(-(N-m)**2/(2*8**2))
    return c

BE_alpha = BE_ld(4, 2)

def Q_with_shell(A, Z):
    N = A - Z
    be_p = BE_ld(A,   Z)   + shell_corr(Z,   N)
    be_d = BE_ld(A-4, Z-2) + shell_corr(Z-2, N-2)
    return be_d + BE_alpha - be_p

Z_range = np.arange(100, 130)
N_range = np.arange(150, 200)
ZZ, NN  = np.meshgrid(Z_range, N_range)
AA      = ZZ + NN

Q_map = np.vectorize(Q_with_shell)(AA, ZZ)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.contourf(Z_range, N_range, Q_map, levels=30, cmap='RdYlGn_r')
plt.colorbar(im, ax=ax, label='Alpha decay Q-value (MeV)')
ax.plot(118, 176, 'w*', markersize=12, label='Og (Z=118, heaviest known)')
ax.plot(114, 184, 'c^', markersize=10, label='Predicted doubly magic (Z=114, N=184)')
ax.plot(120, 184, 'ms', markersize=10, label='Element 120 target (Z=120, N=184)')
ax.axvline(x=114, color='cyan',    linewidth=1, linestyle=':', alpha=0.7)
ax.axvline(x=120, color='magenta', linewidth=1, linestyle=':', alpha=0.7)
ax.axhline(y=184, color='cyan',    linewidth=1, linestyle=':', alpha=0.7)
ax.set_xlabel('Proton number Z', fontsize=12)
ax.set_ylabel('Neutron number N', fontsize=12)
ax.set_title('Alpha decay Q-value in the superheavy region\n'
             'Lower Q = more stable (longer half-life)', fontsize=11)
ax.legend(fontsize=9, loc='upper left')

N_scan  = np.arange(155, 200)
be_ld   = np.array([BE_ld(120+N, 120)/(120+N) for N in N_scan])
be_full = np.array([(BE_ld(120+N,120) + shell_corr(120,N))/(120+N) for N in N_scan])

ax = axes[1]
ax.plot(N_scan, be_ld,   color='steelblue', linewidth=2,   linestyle='--', label='Liquid drop only')
ax.plot(N_scan, be_full, color='firebrick', linewidth=2.5, label='Liquid drop + shell correction')
ax.axvline(x=184, color='gold', linewidth=2, linestyle=':', label='N=184 (predicted magic)')
ax.set_xlabel('Neutron number N (for Z=120)', fontsize=12)
ax.set_ylabel('Binding energy per nucleon (MeV)', fontsize=12)
ax.set_title('Effect of shell correction on element 120\n'
             'Peak at N=184 is the island of stability', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Nuclear shell model and the island of stability (simplified)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
Q_no_shell  = (BE_ld(300, 120) + BE_alpha) - BE_ld(304, 120)  # Q = daughter + alpha - parent
Q_with_sc   = Q_with_shell(304, 120)

print('Element 120, isotope 304-120 (Z=120, N=184):')
print(f'  Q_alpha (liquid drop only):      {Q_no_shell:.2f} MeV')
print(f'  Q_alpha (with shell correction): {Q_with_sc:.2f} MeV')
print()

Z_daughter = 118
a_gamow    = 87 * Z_daughter
ratio_exp  = a_gamow * (1/np.sqrt(Q_with_sc) - 1/np.sqrt(Q_no_shell))

print(f'Estimated half-life enhancement from shell correction:')
print(f'  Exponent: {ratio_exp:.1f}')
print(f'  Factor:   ~10^{ratio_exp/np.log(10):.0f}')
print()
print('Even a few MeV of extra binding from shell corrections can change')
print('half-lives by many orders of magnitude -- this is why the island matters.')

### Questions

1. From the Q-value map: where is the region of lowest Q (most stable) in the superheavy
   chart of nuclides? Does it coincide with Z=114/N=184, Z=120/N=184, or somewhere else?

2. The binding energy plot shows a local maximum at N=184 for Z=120. By how many keV per
   nucleon does the shell correction increase the binding energy at N=184 compared to N=170?
   (Read from the plot.)

3. The calculation shows that even a few MeV reduction in Q_alpha changes the half-life by
   many orders of magnitude. Explain qualitatively why the Gamow tunneling factor is so
   sensitive to small changes in Q. (Hint: think about what determines the tunneling
   probability through the Coulomb barrier.)

4. The shell corrections used here are schematic Gaussians. Real nuclear structure calculations
   (Hartree-Fock-Bogoliubov or relativistic mean-field) give different predictions for whether
   Z=114 or Z=120 is the proton magic number. What experimental observation would definitively
   tell you which prediction is correct?

5. The Ti-50 + Cf-249 reaction produces a compound nucleus with Z=120 and N=179 (not N=184).
   Looking at your binding energy plot, how far is N=179 from the stability maximum at N=184?
   How many neutrons need to be evaporated to reach the most stable isotope, and is this
   achievable in a fusion-evaporation reaction?